# **GENERATE SENSOR DATA**

In [3]:
import numpy as np
import pandas as pd

#Generate sample data
timestamp=pd.date_range(start='2024-01-01',periods=1000,freq='1min')

normal_temp=np.random.uniform(20,80,800)
fault_temp=np.random.uniform(90,100,200)
temperature = np.concatenate([normal_temp, fault_temp])

normal_volt=np.random.uniform(3,5.5,800)
fault_volt1=np.random.uniform(6,7,100)
fault_volt2=np.random.uniform(1,2,100)
voltage=np.concatenate([normal_volt,fault_volt1,fault_volt2])

normal_current=np.random.uniform(0.1,2,800)
fault_current=np.random.uniform(0.01,0.05,200)
current=np.concatenate([normal_current,fault_current])

fault = np.concatenate([np.zeros(800), np.ones(200)])

df = pd.DataFrame({
    'timestamp': timestamp,
    'temperature': temperature,
    'voltage': voltage,
    'current': current,
    'fault': fault
})

print(df.shape)
print("\n Fault distribution:")
print(df['fault'].value_counts())




(1000, 5)

 Fault distribution:
fault
0.0    800
1.0    200
Name: count, dtype: int64


# **TRAIN AND TEST THE MODEL**

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X=df[['temperature','voltage','current']]
y=df['fault']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
model=RandomForestClassifier()
#Train the model
model.fit(X_train,y_train)
#Test the model
y_pred=model.predict(X_test)
#Determine accuracy score
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

1.0


# **DETERMINE THE IMPORTANCE OF EACH CELL**

In [5]:
importances=pd.Series(model.feature_importances_,index=['temperature','voltage','current']).sort_values(ascending=False)
print(importances)

current        0.423247
temperature    0.392416
voltage        0.184338
dtype: float64


# **SAVE DATASET AND MODEL**

In [6]:
import joblib
#Save the model
joblib.dump(model,'fault_detect.pkl')
#Save data
df.to_csv('sensor_data.csv',index=False)
print("Model and dataset saved!")

Model and dataset saved!


In [7]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

# **SEMANTIC SEARCH USING CHROMADB**

In [8]:
import chromadb
client=chromadb.Client()
#Create a collection
collections=client.create_collection('sensor-knowledge')
documents = [
    "A voltage spike occurs when voltage exceeds normal operating range of 3.0-5.5V. This can damage sensitive components and microcontrollers. Immediate action: shut down the system and check the power supply unit.",
    "Temperature overload happens when sensor readings exceed 80 degrees celsius. This can cause component failure and circuit damage. Immediate action: shut down system, improve ventilation, check cooling systems.",
    "Current drop below 0.05A indicates a broken connection, loose wire, or short circuit. This is a critical fault. Immediate action: inspect all connections and replace damaged wires.",
    "Normal sensor operation requires temperature between 20-80°C, voltage between 3.0-5.5V, and current between 0.1-2.0A. Any reading outside these ranges indicates a potential fault.",
    "Voltage drop below 2.0V indicates power supply failure or battery depletion. The system may shut down unexpectedly. Immediate action: check power source and replace if necessary.",
    "Multiple simultaneous faults in temperature and voltage indicate a catastrophic system failure. Immediately shut down all systems and perform full diagnostic before restarting.",
    "Sensor calibration drift occurs when readings gradually shift from normal ranges over time. Regular calibration checks every 30 days are recommended for accurate readings."
]
collections.add(documents=documents,ids=['doc1','doc2','doc3','doc4','doc5','doc6','doc7'])


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 38.3MiB/s]


# **PREDICTION**

In [9]:

new_reading = pd.DataFrame({
    'temperature': [95],  # clearly abnormal
    'voltage': [6.5],     # clearly abnormal
    'current': [0.03]     # clearly abnormal
})
#Predict for some random readings
prediction = model.predict(new_reading)[0]
print("Prediction:", "FAULT" if prediction == 1 else "NORMAL")

Prediction: FAULT


In [10]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


# **GROUNDED EXPLANATION USING GROQ LLaMA API**

In [15]:
from groq import Groq

client=Groq(api_key="YOUR_API_KEY")
def get_completion(prompt,model="llama-3.1-8b-instant"):
  messages=[{"role":"user","content":prompt}]
  response=client.chat.completions.create(model=model,messages=messages,temperature=0)
  return response.choices[0].message.content
if prediction == 1:
  #Find 2 relevant documents
    query = f"temperature is {new_reading['temperature'][0]}°C, voltage is {new_reading['voltage'][0]}V, current is {new_reading['current'][0]}A. What faults are present and what should be done?"
    results=collections.query(query_texts=[query],n_results=2)
    for doc in results['documents'][0]:
      print("-")
      print(doc)
      #Generate explanation
    context="\n".join(results['documents'][0])
    prompt=f"""From the context delimited by triple backticks,find out the appropriate reason for the fault sensor data values
    context=```{context}```
    """
    response=get_completion(prompt)
    print(response)


-
Multiple simultaneous faults in temperature and voltage indicate a catastrophic system failure. Immediately shut down all systems and perform full diagnostic before restarting.
-
Normal sensor operation requires temperature between 20-80°C, voltage between 3.0-5.5V, and current between 0.1-2.0A. Any reading outside these ranges indicates a potential fault.
Based on the provided context, the fault sensor data values are likely due to a catastrophic system failure. The reason for this is that there are multiple simultaneous faults in temperature and voltage, which are outside the normal operating ranges.

Specifically, the faults are:

- Temperature: outside the range of 20-80°C
- Voltage: outside the range of 3.0-5.5V

These simultaneous faults indicate a critical failure of the system, which requires immediate shutdown and a full diagnostic before restarting.


In [16]:
!pip install json

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


# **VALIDATE USING PYDANTIC**

In [18]:
from pydantic import BaseModel
import re,json

class FaultDetection(BaseModel):
  fault_detected:bool
  fault_reason:str
  severity:str
  immediate_action:str

#Generate JSON output
prompt=f"""From the context delimited by triple backticks,analyse the fault data and return only in JSON format with the below fields
-fault_detected(boolean)
-fault_reason(string)
-severity(string)
-immediate_action(string)
context=```{context}```
"""

response = get_completion(prompt)

# Clean response
json_match = re.search(r'\{.*\}', response, re.DOTALL)
if json_match:
    clean_response = json_match.group()
    data = json.loads(clean_response)
    validate = FaultDetection(**data)
    #Print the analysis
    print("Fault Detected:", validate.fault_detected)
    print("Reason:", validate.fault_reason)
    print("Severity:", validate.severity)
    print("Immediate Action:", validate.immediate_action)
else:
    print("Raw response:", response)





Fault Detected: True
Reason: Multiple simultaneous faults in temperature and voltage
Severity: catastrophic
Immediate Action: Immediately shut down all systems and perform full diagnostic before restarting
